# 04 — Modeling

Train a majority-class baseline plus interpretable (logistic regression, decision tree) and ML (random forest, gradient boosting) classifiers. Compare them on accuracy, precision, recall, F1, and ROC-AUC.

**Inputs**: `data/processed/cleaned.csv`  
**Outputs**: results table; optionally serialized models in `models/`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score

from src.config import PROCESSED_DIR, TARGET, RANDOM_STATE
from src.features.build import build_preprocessor
from src.models.train import candidate_models, make_pipeline
from src.models.evaluate import compare

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'cleaned.csv')
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric = X.select_dtypes('number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
print('n_numeric:', len(numeric), '| n_categorical:', len(categorical))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('train:', X_train.shape, '| test:', X_test.shape)

In [ ]:
preprocessor = build_preprocessor(numeric, categorical)
fitted = {}
for name, model in candidate_models().items():
    pipe = make_pipeline(preprocessor, model)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
list(fitted.keys())

In [ ]:
results = compare(fitted, X_test, y_test)
results

In [ ]:
# 5-fold cross-validated ROC-AUC on the training set — sanity check on overfitting.
cv_scores = {}
for name, pipe in fitted.items():
    if name == 'baseline':
        continue
    scores = cross_val_score(pipe, X_train, y_train, scoring='roc_auc', cv=5, n_jobs=-1)
    cv_scores[name] = {'mean_auc': scores.mean(), 'std_auc': scores.std()}
pd.DataFrame(cv_scores).T.sort_values('mean_auc', ascending=False)

## Notes

- The headline number is ROC-AUC vs. the baseline. Models that don't beat the baseline are not predictive.
- For the interpretable models, also report coefficients (`logistic_regression`) or feature importances (`decision_tree`, `random_forest`, `gradient_boosting`) — that's how the proposal's "practical recommendations" objective gets fulfilled.
- Consider calibration (Brier score / calibration curve) before recommending a probability cutoff.
